# Structured Output
<img src="https://github.com/jayyanar/agentic-ai-training/blob/lab-day-1/batch2/lca-langchainV1-essentials/assets/LC_StructuredOutput.png?raw=1" width="500">


## Setup

Load and/or check for needed environmental variables

What we're doing: Install packages required for the structured-output examples in Colab.

In [1]:
!pip install -qU langchain-groq langgraph langchain-community pysqlite3-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


What we're doing: Load the GROQ API key into the environment from Colab userdata.

In [2]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

## Structured Output Example

What we're doing: Initialize the Groq model and define a `TypedDict` response format, then invoke the agent to produce structured output.

In [7]:
from typing_extensions import TypedDict

from langchain.agents import create_agent
from langchain_groq import ChatGroq

# Initialize the Groq model
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_retries=2,
)

class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str
    order: str
    company: str


agent = create_agent(model=llm, response_format=ContactInfo)

recorded_conversation = """We talked with Sachin Chandran. He works over at Wells Fargo. His number is, let's see,
Nine, Eight, Seven, Five, two, three, four, five, six, Seven. Did you get that?
And, his email was Sachinchandran at Wellsfargo.com. He wanted to order 3 pints of Beer."""

result = agent.invoke(
    {"messages": [{"role": "user", "content": recorded_conversation}]}
)

result["structured_response"]

{'name': 'Sachin Chandran',
 'email': 'Sachinchandran@wellsfargo.com',
 'phone': '9875234546',
 'order': '3 pints of Beer',
 'company': 'Wells Fargo'}

#### Multiple data types are supported

* pydantic `BaseModel`
* `TypedDict`
* `dataclasses`
* json schema (dict)

What we're doing: Show an alternative structured output using a `pydantic.BaseModel` as the response schema and invoke the agent.

In [8]:
from langchain.agents import create_agent
from pydantic import BaseModel


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str


agent = create_agent(model=llm, response_format=ContactInfo)

recorded_conversation = """ We talked with Sachin Chandran. He works over at Wells Fargo. His number is, let's see,
Nine, Eight, Seven, Five, two, three, four, five, six, Seven. Did you get that?
And, his email was Sachinchandran at Wellsfargo.com. He wanted to order 3 pints of Beer."""

result = agent.invoke(
    {"messages": [{"role": "user", "content": recorded_conversation}]}
)

result["structured_response"]

ContactInfo(name='Sachin Chandran', email='Sachinchandran@wellsfargo.com', phone='9875234547')